In [171]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from category_encoders import BinaryEncoder
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

In [130]:
df = pd.read_csv('/Users/allikhankoshamet/Desktop/csv/merged_df.csv')
df.head(5)

,name,information,address,price,owner,complex_name,house_type,in_pledge,construction_year,ceiling_height,...,distance_to_botanical_garden,distance_to_triathlon_park,distance_to_astana_park,distance_to_treatment_facility,distance_to_railway_station_1,distance_to_railway_station_2,distance_to_industrial_zone,kzt_sq_m,last_floor,first_floor
0,"3-комнатная квартира, 77 м², 1/4 этаж","2018 г.п., санузел раздельный, ✅Полноценная 3 ...",Е 496 10,33000000,agent,kemel,monolithic,False,2018,2.5,...,3.703569,7.077784,9.179456,10.063298,13.597845,8.466373,11.385081,428571.428571,False,True
1,"1-комнатная квартира, 40 м², 3/14 этаж","жил. комплекс Jetisu.Lepsi, монолитный дом, 20...",Улы Дала — Ұлы дала,21000000,agent,jetisu.lepsi,monolithic,False,2023,3.0,...,4.259391,5.519950,8.360829,11.188389,12.702507,5.137294,8.162841,525000.000000,False,False
2,"1-комнатная квартира, 43.8 м², 6/18 этаж","жил. комплекс BURABAY, монолитный дом, 2022 г....",Ж. Нажимеденова 62 — А62,18500000,agent,burabay,monolithic,False,2022,2.7,...,8.341438,6.716751,9.661737,14.427095,13.096655,0.503983,3.685814,422374.429224,False,False
3,"3-комнатная квартира, 115 м², 5/7 этаж","жил. комплекс Отан 2, кирпичный дом, 2023 г.п....",Байтурсынова 46/2 — BINOM мектебі,51000000,agent,отан,brick,False,2023,2.8,...,7.089332,5.026506,7.936212,12.834399,11.394173,1.272144,3.486966,443478.260870,False,False
4,"2-комнатная квартира, 60 м², 13/14 этаж","жил. комплекс Hazar, монолитный дом, 2022 г.п....",Мангилик Ел 62 — Мангилик Ел и Фариза Онгарсы...,25000000,agent,hazar,monolithic,False,2022,2.7,...,2.443335,5.912639,7.919513,8.999760,12.334288,8.068332,10.760450,416666.666667,False,False


In [131]:
df.columns.tolist()

['name',
 'information',
 'address',
 'price',
 'owner',
 'complex_name',
 'house_type',
 'in_pledge',
 'construction_year',
 'ceiling_height',
 'bathroom_info',
 'condition',
 'area',
 'room_count',
 'floor',
 'floor_count',
 'district',
 'complex_class',
 'parking',
 'elevator',
 'schools_within_500m',
 'kindergartens_within_500m',
 'park_within_1km',
 'coordinates',
 'distance_to_center',
 'distance_to_botanical_garden',
 'distance_to_triathlon_park',
 'distance_to_astana_park',
 'distance_to_treatment_facility',
 'distance_to_railway_station_1',
 'distance_to_railway_station_2',
 'distance_to_industrial_zone',
 'kzt_sq_m',
 'last_floor',
 'first_floor']

In [132]:
df = df.copy()

# убираем мусор
df = df[df["price"] > 0]
df = df[df["area"] > 10]

# target в экономически правильной форме
df["price_per_m2"] = df["price"] / df["area"]
df["log_price_per_m2"] = np.log(df["price_per_m2"])

In [147]:
df.dropna()

,name,information,address,price,owner,complex_name,house_type,in_pledge,construction_year,ceiling_height,...,distance_to_astana_park,distance_to_treatment_facility,distance_to_railway_station_1,distance_to_railway_station_2,distance_to_industrial_zone,kzt_sq_m,last_floor,first_floor,price_per_m2,log_price_per_m2
1,"1-комнатная квартира, 40 м², 3/14 этаж","жил. комплекс Jetisu.Lepsi, монолитный дом, 20...",Улы Дала — Ұлы дала,21000000,agent,jetisu.lepsi,monolithic,False,2023,3.0,...,8.360829,11.188389,12.702507,5.137294,8.162841,525000.000000,False,False,525000.000000,13.171154
5,"1-комнатная квартира, 44 м², 9/13 этаж","В залоге, жил. комплекс Эльнара, монолитный до...",Янушкевича 1/2 — Иманова,18000000,agent,эльнара,monolithic,True,2008,2.7,...,2.776737,8.706765,5.329070,7.276641,6.778492,409090.909091,False,False,409090.909091,12.921693
8,"5-комнатная квартира, 200 м², 2/5 этаж","жил. комплекс Ак Булак, кирпичный дом, 2008 г....",Тасшокы 3,118500000,agent,ак булак,brick,False,2008,3.0,...,2.911360,8.312485,6.684084,6.298450,6.670784,592500.000000,False,False,592500.000000,13.292106
9,"3-комнатная квартира, 93.2 м², 8/9 этаж","жил. комплекс Promenade Expo, монолитный дом, ...",Мәңгілік ел — Ұлы Дала,57000000,agent,promenade expo,monolithic,False,2017,2.7,...,6.834101,8.243716,11.252792,7.715996,10.172978,611587.982833,False,False,611587.982833,13.323814
10,"2-комнатная квартира, 53.1 м², 6/10 этаж","жил. комплекс Apple City, 2020 г.п., состояние...",Бокейхана — Мангилик ел,28500000,agent,apple city,monolithic,False,2020,3.0,...,7.660286,9.036530,12.088796,7.646682,10.318959,536723.163842,False,False,536723.163842,13.193238
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18424,"1-комнатная квартира, 40 м², 8/9 этаж","В залоге, жил. комплекс Nova City, монолитный ...",Аль-фараби 34/4 — возле школы BINOM,22000000,owner,nova city,monolithic,True,2019,2.7,...,7.850685,10.186589,12.265004,6.103304,8.945611,550000.000000,False,False,550000.000000,13.217674
18426,"1-комнатная квартира, 32 м², 10/12 этаж","жил. комплекс Изобилие, кирпичный дом, 2010 г....",Жанибека Тархана 9 — Бактыораза Бейсекбаева,15000000,agent,изобилие,monolithic,False,2010,2.7,...,2.276105,8.214581,4.898878,7.738491,7.319381,468750.000000,False,False,468750.000000,13.057825
18427,"2-комнатная квартира, 73.1 м², 9/14 этаж","В залоге, жил. комплекс Биик Шанырак, монолитн...",Ракымжана Кошкарбаева,28000000,owner,биик шанырак,monolithic,True,2019,2.7,...,6.423164,11.676379,9.687050,2.915104,3.526616,383036.935705,False,False,383036.935705,12.855887
18429,"2-комнатная квартира, 70 м², 7/9 этаж","кирпичный дом, 2007 г.п., состояние: хорошее, ...",Жирентаева 2 — Жирентаева-Майлина,37000000,owner,анфилада,brick,False,2007,2.7,...,4.147250,9.803751,7.221142,5.386784,5.224939,528571.428571,False,False,528571.428571,13.177933


## Place tax 


In [133]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, r2_score

In [134]:
CATEGORICAL_FEATURES = [
    "condition",
    "house_type",
    "complex_class",
    "bathroom_info",
    "owner",
    "parking",
    "elevator",
    "in_pledge"
]

In [135]:
BASE_FEATURES = [
    "area",
    "room_count",
    "floor",
    "floor_count",
    "construction_year",
    "ceiling_height",
    "first_floor",
    "last_floor",

    # categorical
    "condition",
    "house_type",
    "complex_class",
    "bathroom_info",
    "owner",
    "parking",
    "elevator",
    "in_pledge"
]

In [136]:
LOCATION_FEATURES = [
    "district",
    "schools_within_500m",
    "kindergartens_within_500m",
    "park_within_1km",
    "distance_to_center",
    "distance_to_botanical_garden",
    "distance_to_triathlon_park",
    "distance_to_astana_park",
    "distance_to_treatment_facility",
    "distance_to_railway_station_1",
    "distance_to_railway_station_2",
    "distance_to_industrial_zone"
]

In [148]:
df = df.dropna(subset=BASE_FEATURES + LOCATION_FEATURES + ['log_price_per_m2'])  # Explicitly drop on relevant columns
X_base = df[BASE_FEATURES]
# ... (split and Pool)

In [149]:
X_base = df[BASE_FEATURES]
X_full = df[BASE_FEATURES + LOCATION_FEATURES]
y = df["log_price_per_m2"]

In [150]:
cat_features_base = [c for c in CATEGORICAL_FEATURES if c in BASE_FEATURES]
cat_features_full = [c for c in CATEGORICAL_FEATURES if c in (BASE_FEATURES + LOCATION_FEATURES)]

In [151]:
Xb_train, Xb_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

Xf_train, Xf_test, _, _ = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)

In [152]:
for col in cat_features_base:
    Xb_train[col] = Xb_train[col].astype(str)
    Xb_test[col] = Xb_test[col].astype(str)  # Do the same for test

In [153]:
from catboost import Pool

train_pool = Pool(
    data=Xb_train,
    label=y_train,
    cat_features=cat_features_base,
    feature_names=Xb_train.columns.tolist()
)

model_base = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False
)

model_base.fit(train_pool)

In [154]:
from catboost import CatBoostRegressor

model_base = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

model_base.fit(train_pool)

# Price for each parameter 


In [ ]:
import numpy as np

shap_values = model_base.get_feature_importance(train_pool, type='ShapValues')[:, :-1]  # Exclude bias term
mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
mean_price_per_m2 = np.mean(np.exp(y_train))

# KZT per m²
monetary_impacts = mean_abs_shap * mean_price_per_m2

# Create a DataFrame for readability
fi_money = pd.DataFrame({
    'Feature': Xb_train.columns.tolist(),
    'Monetary Impact (KZT per m²)': monetary_impacts.round(2)
}).sort_values('Monetary Impact (KZT per m²)', ascending=False)

fi_money.head(15)

,Feature,Monetary Impact (KZT per m²)
8,condition,31321.96
4,construction_year,27628.11
10,complex_class,26595.28
5,ceiling_height,26059.63
12,owner,14829.47
0,area,13294.54
9,house_type,11608.35
13,parking,11568.23
3,floor_count,9024.09
1,room_count,8811.56


## Overrated

In [102]:
top_overpriced = (
    data.sort_values("overpricing_pct", ascending=False)
      .head(10)[
          [
              "name",
              "district",
              "area",
              "room_count",
              "floor",
              "actual_price",
              "fair_price",
              "overpricing_pct"
          ]
      ]
)

top_overpriced

,name,district,area,room_count,floor,actual_price,fair_price,overpricing_pct
6303,"2-комнатная квартира, 52.9 м², 13/14 этаж",saryarka,52.9,2,13.0,240000000.0,2.218285e+07,9.819167
5474,"3-комнатная квартира, 120 м², 10/13 этаж",esil,120.0,3,10.0,170000000.0,6.630033e+07,1.564090
7043,"4-комнатная квартира, 107 м², 5/14 этаж",esil,107.0,4,5.0,141000000.0,6.772523e+07,1.081942
7656,"4-комнатная квартира, 120 м², 4/16 этаж",saryarka,120.0,4,4.0,105000000.0,5.221635e+07,1.010864
2986,"3-комнатная квартира, 220 м², 3/5 этаж",nura,220.0,3,3.0,315000000.0,1.643542e+08,0.916592
6477,"4-комнатная квартира, 150 м², 7/8 этаж",esil,150.0,4,7.0,190000000.0,1.025536e+08,0.852690
545,"3-комнатная квартира, 130 м², 8/8 этаж",esil,130.0,3,8.0,159999999.0,8.814664e+07,0.815157
7601,"4-комнатная квартира, 114 м², 9/15 этаж",saryarka,114.0,4,9.0,96500000.0,5.444883e+07,0.772306
7718,"4-комнатная квартира, 115 м², 15/16 этаж",saryarka,115.0,4,15.0,99000000.0,5.651632e+07,0.751707
7562,"1-комнатная квартира, 43 м², 4/12 этаж",saryarka,43.0,1,4.0,24500000.0,1.404311e+07,0.744628


## Underrated 

In [103]:
top_underpriced = (
    data.sort_values("overpricing_pct")
      .head(10)[
          [
              "name",
              "district",
              "area",
              "room_count",
              "floor",
              "actual_price",
              "fair_price",
              "overpricing_pct"
          ]
      ]
)

top_underpriced

,name,district,area,room_count,floor,actual_price,fair_price,overpricing_pct
7767,"1-комнатная квартира, 38.08 м², 12/12 этаж",almaty,38.08,1,12.0,4000000.0,1.100156e+07,-0.636415
4117,"2-комнатная квартира, 50.2 м², 1/12 этаж",almaty,50.20,2,1.0,10500000.0,2.201700e+07,-0.523096
2730,"2-комнатная квартира, 50 м², 1/12 этаж",saryarka,50.00,2,1.0,11000000.0,1.887401e+07,-0.417188
1751,"3-комнатная квартира, 102.9 м², 12/19 этаж",esil,102.90,3,12.0,38000000.0,6.197901e+07,-0.386889
6648,"2-комнатная квартира, 69.2 м², 8/12 этаж",baikonur,69.20,2,8.0,14800000.0,2.412141e+07,-0.386437
6409,"2-комнатная квартира, 65 м², 4/7 этаж",almaty,65.00,2,4.0,14500000.0,2.319761e+07,-0.374936
3545,"3-комнатная квартира, 90 м², 12/12 этаж",esil,90.00,3,12.0,32000000.0,5.108513e+07,-0.373595
970,"4-комнатная квартира, 125.4 м², 3/12 этаж",nura,125.40,4,3.0,44000000.0,7.015649e+07,-0.372831
3535,"2-комнатная квартира, 65 м², 4/7 этаж",almaty,65.00,2,4.0,14500000.0,2.210896e+07,-0.344157
2816,"2-комнатная квартира, 40 м², 4/4 этаж",nura,40.00,2,4.0,9500000.0,1.445939e+07,-0.342988


## Market stats

In [104]:
pricing_stats = (
    data["pricing_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

pricing_stats

pricing_label
Fair price             84.51
Overpriced             10.42
Underpriced             3.21
Strongly overpriced     1.87
Name: proportion, dtype: float64

## Owner or Realtor

In [105]:
owner_vs_realtor = (
    data.groupby("owner")["overpricing_pct"]
      .median()
      .sort_values(ascending=False)
)

owner_vs_realtor

owner
owner   -0.000819
agent   -0.002661
Name: overpricing_pct, dtype: float64

In [163]:
from sklearn.model_selection import KFold

In [164]:
q_low = df['price_per_m2'].quantile(0.01)
q_high = df['price_per_m2'].quantile(0.99)
df = df[(df['price_per_m2'] > q_low) & (df['price_per_m2'] < q_high)]
print(f'New shape after outlier removal: {df.shape}')

New shape after outlier removal: (9864, 41)


In [165]:
LOCATION_FEATURES = [
    'distance_to_center', 'distance_to_botanical_garden', 'distance_to_triathlon_park',
    'distance_to_astana_park', 'distance_to_treatment_facility', 'distance_to_railway_station_1',
    'distance_to_railway_station_2', 'distance_to_industrial_zone'
]
enhanced_features = BASE_FEATURES + LOCATION_FEATURES

In [166]:
for col in cat_features_base:
    df[col] = df[col].fillna('missing').astype(str)

df = df.dropna(subset=enhanced_features + ['log_price_per_m2'])

In [167]:
X_enhanced = df[enhanced_features]
y = df['log_price_per_m2']
Xb_train, Xb_test, y_train, y_test = train_test_split(X_enhanced, y, test_size=0.2, random_state=42)

train_pool = Pool(data=Xb_train, label=y_train, cat_features=cat_features_base)
test_pool = Pool(data=Xb_test, label=y_test, cat_features=cat_features_base)

In [168]:
model_enhanced = CatBoostRegressor(
    iterations=1000,  # Больше итераций
    depth=8,  # Глубже
    learning_rate=0.03,  # Медленнее
    loss_function='MAE',  # Robust к outliers
    verbose=False,
    random_seed=42
)

In [169]:
kf = KFold(n_splits=5)
rmse_scores = []
for train_idx, val_idx in kf.split(Xb_train):
    train_sub = Pool(Xb_train.iloc[train_idx], y_train.iloc[train_idx], cat_features=cat_features_base)
    val_sub = Pool(Xb_train.iloc[val_idx], y_train.iloc[val_idx], cat_features=cat_features_base)
    model_enhanced.fit(train_sub)
    preds_val = model_enhanced.predict(val_sub)
    rmse_scores.append(np.sqrt(mean_squared_error(y_train.iloc[val_idx], preds_val)))
print(f'CV RMSE (log): {np.mean(rmse_scores):.4f}')

CV RMSE (log): 0.1234


In [172]:
model_enhanced.fit(train_pool)

# Шаг 5: Метрики на тесте
preds_log_test = model_enhanced.predict(test_pool)
rmse_log = np.sqrt(mean_squared_error(y_test, preds_log_test))
mae_log = mean_absolute_error(y_test, preds_log_test)

# В KZT scale
y_test_kzt = np.exp(y_test)
preds_kzt = np.exp(preds_log_test)
rmse_kzt = np.sqrt(mean_squared_error(y_test_kzt, preds_kzt))
mae_kzt = mean_absolute_error(y_test_kzt, preds_kzt)
mape = np.mean(np.abs((y_test_kzt - preds_kzt) / y_test_kzt)) * 100

In [173]:
print(f'RMSE (log scale): {rmse_log:.4f}')
print(f'MAE (log scale): {mae_log:.4f}')
print(f'RMSE (KZT/m²): {rmse_kzt:.0f}')
print(f'MAE (KZT/m²): {mae_kzt:.0f}')
print(f'MAPE (% error): {mape:.2f}%')

RMSE (log scale): 0.1231
MAE (log scale): 0.0931
RMSE (KZT/m²): 65372
MAE (KZT/m²): 47059
MAPE (% error): 9.31%


In [174]:
preds_log_full = model_enhanced.predict(X_enhanced)
df['fair_price_per_m2'] = np.exp(preds_log_full)
df['fair_price'] = df['fair_price_per_m2'] * df['area']
df['overpricing_pct'] = ((df['price'] - df['fair_price']) / df['fair_price']) * 100

In [175]:
top_overpriced = df.sort_values('overpricing_pct', ascending=False).head(10)[
    ['name', 'district', 'area', 'room_count', 'floor', 'price', 'fair_price', 'overpricing_pct']
].rename(columns={'price': 'actual_price'})

top_underpriced = df.sort_values('overpricing_pct', ascending=True).head(10)[
    ['name', 'district', 'area', 'room_count', 'floor', 'price', 'fair_price', 'overpricing_pct']
].rename(columns={'price': 'actual_price'})

print("Top Overpriced Apartments:")
print(top_overpriced)

print("\nTop Underpriced Apartments:")
print(top_underpriced)

Top Overpriced Apartments:
                                           name  district   area  room_count  \
15058   6-комнатная квартира, 417 м², 9/12 этаж      esil  417.0           6   
13949   4-комнатная квартира, 120 м², 4/16 этаж  saryarka  120.0           4   
13832   4-комнатная квартира, 114 м², 9/15 этаж  saryarka  114.0           4   
14068  4-комнатная квартира, 115 м², 15/16 этаж  saryarka  115.0           4   
5508    2-комнатная квартира, 77 м², 13/18 этаж    almaty   77.0           2   
6372   4-комнатная квартира, 168 м², 17/20 этаж  baikonur  168.0           4   
8632     3-комнатная квартира, 68 м², 9/17 этаж      nura   68.0           3   
8175     2-комнатная квартира, 66 м², 8/10 этаж    almaty   66.0           2   
14399    3-комнатная квартира, 180 м², 7/9 этаж  saryarka  180.0           3   
13733  5-комнатная квартира, 116 м², 16/16 этаж  saryarka  116.0           5   

       floor  actual_price    fair_price  overpricing_pct  
15058    9.0     390000000  1.89

In [179]:
avg_price_by_district = df.groupby('district')['price_per_m2'].mean().sort_values(ascending=False).round(0)

# Most expensive district
most_expensive_district = avg_price_by_district.index[0]
most_expensive_avg = avg_price_by_district.iloc[0]

print("Average Price per m² by District (KZT):")
print(avg_price_by_district)

print(f"\nThe most expensive district is '{most_expensive_district}' with average {most_expensive_avg:.0f} KZT per m².")

Average Price per m² by District (KZT):
district
esil         564396.0
nura         539089.0
almaty       465306.0
saryarka     431251.0
Косшы р-н    431049.0
baikonur     426663.0
Name: price_per_m2, dtype: float64

The most expensive district is 'esil' with average 564396 KZT per m².
